# Capstone Analysis Notebook (Reordered and Cleaned)

This notebook is organized as a reproducible end-to-end workflow for the VAERS COVID-19 capstone project.

## Major sections
1. Setup and data loading  
2. COVID-19 subsetting and case-level aggregation  
3. Outcome construction and structured feature engineering  
4. Exploratory data analysis  
5. Rule-based comorbidity indicators  
6. Symptom text preprocessing  
7. Unsupervised clustering (phenotype discovery)  
8. Supervised severity prediction  
9. Saving final artifacts  

> **Important:** Run the notebook from top to bottom after restarting the kernel.  
> Several earlier inconsistencies were caused by out-of-order execution and reuse of stale saved matrices.

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

import os
import re
import sys
import json
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import scipy.sparse as sp

warnings.filterwarnings("ignore")

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

In [ ]:
# ============================================================
# 2. FILE PATHS
# ============================================================

# Update these paths if needed
DATA_PATH = "/Users/ariellerothman/Desktop/Capstone/COMBINED_DATA_ALL_YEARS.csv"
OUTDIR = "/Users/ariellerothman/Desktop/Capstone/Outputs"
os.makedirs(OUTDIR, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTDIR:", OUTDIR)

## 2. Load raw data and inspect structure

In [ ]:
# ============================================================
# 3. LOAD RAW DATA
# ============================================================

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Raw shape:", df_raw.shape)
print(df_raw.head(3))
print("\nColumns:")
print(df_raw.columns.tolist())

In [ ]:
# ============================================================
# 4. RAW DATA TYPES + MISSINGNESS SNAPSHOT
# ============================================================

print(df_raw.info())

missing_pct_raw = (df_raw.isna().mean() * 100).sort_values(ascending=False)
print("\nTop missingness percentages:")
print(missing_pct_raw.head(20))

## 3. Subset to COVID-19 reports and collapse to case level

This section:
- subsets to `VAX_TYPE == "COVID19"`
- removes exact duplicate rows
- collapses multiple row-level records per `VAERS_ID`
- preserves clinically relevant information using conservative aggregation rules

In [ ]:
# ============================================================
# 5. SUBSET TO COVID-19 REPORTS
# ============================================================

df_covid_rows = df_raw.loc[df_raw["VAX_TYPE"] == "COVID19"].copy()

print("COVID row-level shape:", df_covid_rows.shape)
print("Unique VAERS_ID:", df_covid_rows["VAERS_ID"].nunique())
print("Extra rows beyond one-per-ID:", len(df_covid_rows) - df_covid_rows["VAERS_ID"].nunique())

In [ ]:
# ============================================================
# 6. REMOVE EXACT DUPLICATE ROWS
# ============================================================

before = len(df_covid_rows)
df_covid_rows = df_covid_rows.drop_duplicates()
after = len(df_covid_rows)

print("Rows before exact duplicate removal:", before)
print("Rows after exact duplicate removal:", after)
print("Exact duplicates removed:", before - after)

In [ ]:
# ============================================================
# 7. BASIC TYPE CLEANING BEFORE CASE-LEVEL AGGREGATION
# ============================================================

date_cols = ["RECVDATE", "VAX_DATE", "ONSET_DATE", "DATEDIED"]
for col in date_cols:
    if col in df_covid_rows.columns:
        df_covid_rows[col] = pd.to_datetime(df_covid_rows[col], errors="coerce")

# Binary outcome source columns in VAERS are usually Y/blank
binary_source_cols = [
    "DIED", "L_THREAT", "ER_VISIT", "ER_ED_VISIT",
    "HOSPITAL", "DISABLE", "BIRTH_DEFECT"
]

for col in binary_source_cols:
    if col in df_covid_rows.columns:
        df_covid_rows[col] = df_covid_rows[col].astype(str).str.upper().str.strip()
        df_covid_rows[col] = df_covid_rows[col].replace({"Y": 1, "N": 0, "": np.nan, "NAN": np.nan})
        df_covid_rows[col] = pd.to_numeric(df_covid_rows[col], errors="coerce")

# Numeric columns
numeric_cols = ["AGE_YRS", "HOSPDAYS", "NUMDAYS"]
for col in numeric_cols:
    if col in df_covid_rows.columns:
        df_covid_rows[col] = pd.to_numeric(df_covid_rows[col], errors="coerce")

# Standardize basic categorical text
for col in ["SEX", "STATE", "VAX_TYPE", "VAX_MANU", "VAX_NAME", "VAX_DOSE_SERIES", "VAX_ROUTE", "VAX_SITE"]:
    if col in df_covid_rows.columns:
        df_covid_rows[col] = df_covid_rows[col].astype("string")

In [ ]:
# ============================================================
# 8. CASE-LEVEL AGGREGATION
# ============================================================

text_fields = ["SYMPTOM_TEXT", "LAB_DATA", "OTHER_MEDS", "CUR_ILL", "HISTORY", "PRIOR_VAX", "ALLERGIES"]
list_fields = ["VAX_MANU", "VAX_NAME", "VAX_DOSE_SERIES", "VAX_ROUTE", "VAX_SITE", "VAX_LOT"]

def concat_unique_text(series):
    vals = [str(x).strip() for x in series.dropna().astype(str) if str(x).strip()]
    vals_unique = list(dict.fromkeys(vals))
    return " | ".join(vals_unique) if vals_unique else np.nan

def unique_list(series):
    vals = [str(x).strip() for x in series.dropna().astype(str) if str(x).strip()]
    return list(dict.fromkeys(vals))

agg_dict = {}

# Binary outcomes: use max to preserve any serious signal
for col in binary_source_cols:
    if col in df_covid_rows.columns:
        agg_dict[col] = "max"

# Dates
if "RECVDATE" in df_covid_rows.columns:
    agg_dict["RECVDATE"] = "min"
if "VAX_DATE" in df_covid_rows.columns:
    agg_dict["VAX_DATE"] = "min"
if "ONSET_DATE" in df_covid_rows.columns:
    agg_dict["ONSET_DATE"] = "max"
if "DATEDIED" in df_covid_rows.columns:
    agg_dict["DATEDIED"] = "max"

# Numeric vars
if "AGE_YRS" in df_covid_rows.columns:
    agg_dict["AGE_YRS"] = "max"
if "HOSPDAYS" in df_covid_rows.columns:
    agg_dict["HOSPDAYS"] = "max"
if "NUMDAYS" in df_covid_rows.columns:
    agg_dict["NUMDAYS"] = "max"

# Simple categoricals (use first non-null)
for col in ["SEX", "STATE", "VAX_TYPE"]:
    if col in df_covid_rows.columns:
        agg_dict[col] = lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan

# Text fields
for col in text_fields:
    if col in df_covid_rows.columns:
        agg_dict[col] = concat_unique_text

# Exposure list fields
for col in list_fields:
    if col in df_covid_rows.columns:
        agg_dict[col] = unique_list

df_case = df_covid_rows.groupby("VAERS_ID", as_index=False).agg(agg_dict)

print("Case-level shape:", df_case.shape)
print(df_case.head(2).T)

## 4. Outcome construction and structured feature engineering

In [ ]:
# ============================================================
# 9. OUTCOME CONSTRUCTION
# ============================================================

severity_components = [
    "DIED", "DISABLE", "L_THREAT", "HOSPITAL", "ER_VISIT", "ER_ED_VISIT", "BIRTH_DEFECT"
]
severity_components = [c for c in severity_components if c in df_case.columns]

for col in severity_components:
    df_case[col] = df_case[col].fillna(0).astype("int8")

# Composite serious outcome
df_case["SERIOUS"] = (df_case[severity_components].sum(axis=1) > 0).astype("int8")

# Optional severe alias if you want a separate label
df_case["SEVERE"] = df_case["SERIOUS"].copy()

print(df_case[severity_components + ["SERIOUS"]].mean().sort_values(ascending=False))

In [ ]:
# ============================================================
# 10. STRUCTURED FEATURE ENGINEERING
# ============================================================

# Remove implausible ages
if "AGE_YRS" in df_case.columns:
    df_case.loc[(df_case["AGE_YRS"] < 0) | (df_case["AGE_YRS"] > 110), "AGE_YRS"] = np.nan

# Onset interval
if "ONSET_DATE" in df_case.columns and "VAX_DATE" in df_case.columns:
    df_case["ONSET_INTERVAL"] = (df_case["ONSET_DATE"] - df_case["VAX_DATE"]).dt.days

# Temporal features
if "RECVDATE" in df_case.columns:
    df_case["YEAR"] = df_case["RECVDATE"].dt.year.astype("Int64")
    df_case["MONTH"] = df_case["RECVDATE"].dt.month.astype("Int64")

# Dose features
def dose_features(dose_list):
    if not isinstance(dose_list, list) or len(dose_list) == 0:
        return pd.Series([pd.NA, 0, 0, 0], index=["MAX_DOSE","DOSE_COUNT","MULTI_DOSE","UNKNOWN_DOSE"])

    cleaned = []
    unknown = 0
    for d in dose_list:
        if pd.isna(d):
            continue
        d = str(d).strip().upper()
        if d in ("UNK", "UNKNOWN"):
            unknown = 1
            cleaned.append("UNK")
        elif d == "7+":
            cleaned.append("7+")
        else:
            cleaned.append(d)

    nums = []
    for d in cleaned:
        if d == "7+":
            nums.append(7)
        else:
            try:
                nums.append(int(d))
            except Exception:
                pass

    max_dose = max(nums) if nums else pd.NA
    dose_count = len(set(cleaned))
    multi_dose = int(dose_count > 1)

    return pd.Series([max_dose, dose_count, multi_dose, unknown], index=["MAX_DOSE","DOSE_COUNT","MULTI_DOSE","UNKNOWN_DOSE"])

if "VAX_DOSE_SERIES" in df_case.columns:
    df_case[["MAX_DOSE","DOSE_COUNT","MULTI_DOSE","UNKNOWN_DOSE"]] = df_case["VAX_DOSE_SERIES"].apply(dose_features)

# Manufacturer multi-hot
if "VAX_MANU" in df_case.columns:
    manu_lists = df_case["VAX_MANU"].apply(lambda x: x if isinstance(x, list) else [])
    manufacturers = sorted(set(m for lst in manu_lists for m in lst if pd.notna(m)))
    for manu in manufacturers:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", str(manu)).strip("_").upper()
        colname = f"MANU__{safe}"
        df_case[colname] = manu_lists.apply(lambda lst: int(manu in set(lst))).astype("int8")
    df_case["MULTI_MANUFACTURER"] = manu_lists.apply(lambda lst: int(len(set(lst)) > 1)).astype("int8")

print("Manufacturer columns:", [c for c in df_case.columns if c.startswith("MANU__")])
print(df_case[["MAX_DOSE","DOSE_COUNT","MULTI_DOSE","UNKNOWN_DOSE","MULTI_MANUFACTURER"]].head())

In [ ]:
# ============================================================
# 11. MISSINGNESS INDICATORS + BASIC IMPUTATION PREP
# ============================================================

# Add missingness indicators for continuous variables likely to be used in supervised models
for col in ["AGE_YRS", "HOSPDAYS", "NUMDAYS", "ONSET_INTERVAL", "MAX_DOSE"]:
    if col in df_case.columns:
        df_case[f"{col}_MISSING"] = df_case[col].isna().astype("int8")

df_clean = df_case.copy()
print("Working dataframe shape:", df_clean.shape)

## 5. Exploratory data analysis

In [ ]:
# ============================================================
# 12. BASIC DESCRIPTIVE STATISTICS
# ============================================================

print("Case-level dataset shape:", df_clean.shape)
print("\nSerious rate:", df_clean["SERIOUS"].mean())

if "SEX" in df_clean.columns:
    print("\nSex distribution:")
    print(df_clean["SEX"].value_counts(dropna=False))

if "AGE_YRS" in df_clean.columns:
    print("\nAge summary:")
    print(df_clean["AGE_YRS"].describe())

In [ ]:
# ============================================================
# 13. MISSINGNESS SUMMARY
# ============================================================

missing_pct = (df_clean.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(30))

In [ ]:
# ============================================================
# 14. SYMPTOM NARRATIVE LENGTH
# ============================================================

if "SYMPTOM_TEXT" in df_clean.columns:
    df_clean["SYMPTOM_TEXT_CHAR_LEN"] = df_clean["SYMPTOM_TEXT"].fillna("").astype(str).str.len()
    print(df_clean["SYMPTOM_TEXT_CHAR_LEN"].describe())

    plt.figure(figsize=(8,5))
    plt.hist(df_clean["SYMPTOM_TEXT_CHAR_LEN"], bins=100)
    plt.xlim(0, 2500)
    plt.title("Distribution of SYMPTOM_TEXT character length (zoomed to 0–2500)")
    plt.xlabel("Character length")
    plt.ylabel("Count")
    plt.show()

## 6. Rule-based comorbidity indicators

This section converts selected free-text fields into structured binary indicators using field-specific keyword dictionaries.

In [ ]:
# ============================================================
# 15. RULE-BASED COMORBIDITY INDICATORS
# ============================================================

FREE_TEXT_FIELDS = ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]

NULL_TOKENS = {
    "none", "no", "n", "na", "n/a", "unknown", "unk", "none known", "not known",
    "nka", "nkda", "no known allergies", "no known drug allergies",
    "denies", "denies allergies", "no allergies", "nil", "negative"
}

_keep = re.compile(r"[^a-z0-9\s\-\/]")
_ws = re.compile(r"\s+")
_url = re.compile(r"http\S+|www\S+|https\S+")
_email = re.compile(r"\S+@\S+")

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if not x:
        return ""
    if x in NULL_TOKENS:
        return ""
    x = _url.sub(" ", x)
    x = _email.sub(" ", x)
    x = _keep.sub(" ", x)
    x = _ws.sub(" ", x).strip()
    if x in NULL_TOKENS:
        return ""
    return x

CATEGORY_MAP = {
    "CUR_ILL": {
        "acute_infection": [r"\bcovid\b", r"\bsars[-\s]?cov[-\s]?2\b", r"\binfluenza\b", r"\bflu\b", r"\binfection\b", r"\bpneumonia\b", r"\bbronchitis\b", r"\bviral\b", r"\bbacterial\b"],
        "chronic_cardiovascular": [r"\bhypertension\b", r"\bhtn\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b", r"\bheart failure\b", r"\bchf\b", r"\barrhythmia\b", r"\bmi\b", r"\bmyocardial infarction\b"],
        "chronic_respiratory": [r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b", r"\bsleep apnea\b", r"\bosa\b"],
        "metabolic_endocrine": [r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\btype\s?2\b", r"\bobesity\b", r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"],
        "endocrine_thyroid": [r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b", r"\bhashimoto\w*\b", r"\bgraves\b"],
        "autoimmune_inflammatory": [r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b", r"\brheumatoid arthritis\b", r"\bmultiple sclerosis\b", r"\bms\b", r"\bibs\b", r"\bibd\b", r"\bceliac\b"],
        "neurologic": [r"\bseizure\b", r"\bepilepsy\b", r"\bstroke\b", r"\btia\b", r"\bparkinson\w*\b", r"\bneuropath\w*\b", r"\bdementia\b", r"\balzheimer\w*\b"],
        "neuro_headache": [r"\bmigraine\w*\b", r"\bheadache\w*\b", r"\bcluster headache\b"],
        "cancer_immunocompromised": [r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b", r"\btransplant\b", r"\bhiv\b", r"\baids\b", r"\bimmunosupp\w*\b", r"\bimmunocompromis\w*\b"],
        "lymphatic_edema": [r"\blymph(edema|oedema)\b", r"\bedema\b", r"\bswelling\b"],
    },
    "HISTORY": {
        "cardiovascular": [r"\bhtn\b", r"\bhypertension\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b", r"\bheart failure\b", r"\bchf\b", r"\bmi\b", r"\bmyocardial infarction\b", r"\bvalve\b", r"\baortic stenosis\b"],
        "respiratory": [r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b", r"\bosa\b", r"\bsleep apnea\b"],
        "metabolic": [r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\bobesity\b", r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"],
        "endocrine_thyroid": [r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b", r"\bhashimoto\w*\b", r"\bgraves\b"],
        "musculoskeletal": [r"\barthritis\b", r"\bosteoarthritis\b", r"\boa\b", r"\bgout\b", r"\bfibromyalgia\b", r"\bchronic pain\b"],
        "mental_health": [r"\bdepress\w*\b", r"\banxiety\b", r"\bptsd\b", r"\bbipolar\b", r"\bschizo\w*\b"],
        "gastrointestinal": [r"\bgerd\b", r"\bacid reflux\b", r"\bgastroesophageal\b", r"\bibs\b", r"\bconstipat\w*\b", r"\bdiverticul\w*\b"],
        "kidney": [r"\bckd\b", r"\bchronic kidney\b", r"\brenal\b", r"\bdialysis\b", r"\brenal failure\b", r"\bkidney transplant\b"],
        "clotting": [r"\bdvt\b", r"\bpe\b", r"\bthromb\w*\b", r"\bclot\b", r"\bembol\w*\b"],
        "autoimmune": [r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b", r"\bms\b", r"\bmultiple sclerosis\b"],
        "cancer_immunocompromised": [r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b", r"\btransplant\b", r"\bhiv\b", r"\baids\b", r"\bimmunosupp\w*\b"],
        "neurologic": [r"\bstroke\b", r"\btia\b", r"\bseizure\b", r"\bepilepsy\b", r"\bmigraine\w*\b", r"\bneuropath\w*\b", r"\bparkinson\w*\b"],
        "lymphatic_edema": [r"\blymph(edema|oedema)\b", r"\bedema\b"],
    },
    "ALLERGIES": {
        "drug_allergy": [r"\bpenicillin\b", r"\bsulfa\b", r"\bnsaid\b", r"\baspirin\b", r"\bceclor\b", r"\bkeflex\b", r"\bdoxycycline\b"],
        "food_allergy": [r"\bpeanut\b", r"\bnut\b", r"\begg\b", r"\bmilk\b", r"\bshellfish\b", r"\bgarlic\b"],
        "latex_other_contact": [r"\blatex\b", r"\bdetergent\b", r"\bfragrance\b"],
        "anaphylaxis_history": [r"\banaphylax\w*\b", r"\bepi[-\s]?pen\b", r"\bepinephrine\b"],
    },
    "PRIOR_VAX": {
        "prior_covid_vax": [r"\bcovid\b", r"\bpfizer\b", r"\bmoderna\b", r"\bjanssen\b", r"\bnovavax\b"],
        "prior_flu_vax": [r"\bflu\b", r"\binfluenza\b"],
        "prior_other_vax": [r"\bshingrix\b", r"\bshingles\b", r"\btetanus\b", r"\brabies\b", r"\bh1n1\b", r"\bmmr\b", r"\bhepatitis\b", r"\bvaricella\b"],
        "prior_vax_reaction": [r"\breaction\b", r"\ballergic\b", r"\banaphylax\w*\b", r"\bside effect\b", r"\bsyncope\b"],
    },
    "LAB_DATA": {
        "cardiac_markers": [r"\btroponin\b", r"\bbnp\b", r"\bck[-\s]?mb\b"],
        "coagulation": [r"\bd[-\s]?dimer\b", r"\binr\b", r"\bptt\b", r"\bfibrinogen\b"],
        "inflammation": [r"\bcrp\b", r"\besr\b", r"\bferritin\b"],
        "cbc": [r"\bwbc\b", r"\bplatelet\w*\b", r"\bhemoglobin\b", r"\bhgb\b"],
        "vitals_procedures": [r"\bbp\b", r"\bhr\b", r"\bspo2\b", r"\bultrasound\b", r"\bct\b", r"\bmri\b", r"\bx[-\s]?ray\b"],
    },
    "OTHER_MEDS": {
        "anticoagulant": [r"\bwarfarin\b", r"\bheparin\b", r"\beliquis\b", r"\bxarelto\b", r"\bapixaban\b", r"\brivaroxaban\b"],
        "statin": [r"\bstatin\b", r"\batorvastatin\b", r"\bsimvastatin\b", r"\brosuvastatin\b"],
        "immunosuppressant": [r"\bprednisone\b", r"\bsteroid\b", r"\bmethotrexate\b", r"\btacrolimus\b", r"\bcyclosporine\b"],
        "diabetes_meds": [r"\binsulin\b", r"\bmetformin\b", r"\bsemaglutide\b", r"\bozempic\b", r"\bglp[-\s]?1\b"],
        "thyroid_meds": [r"\blevothyroxine\b", r"\bsynthroid\b", r"\bliothyronine\b"],
        "psych_meds": [r"\bsertraline\b", r"\bfluoxetine\b", r"\bcitalopram\b", r"\bescitalopram\b", r"\bbuspirone\b", r"\balprazolam\b", r"\btrazodone\b"],
    },
}

def compile_map(category_map):
    compiled = {}
    for col, cats in category_map.items():
        compiled[col] = {cat: [re.compile(p) for p in pats] for cat, pats in cats.items()}
    return compiled

COMPILED_MAP = compile_map(CATEGORY_MAP)

def flag_any(text, patterns):
    if not text:
        return 0
    return int(any(p.search(text) for p in patterns))

for col in FREE_TEXT_FIELDS:
    if col not in df_clean.columns:
        continue

    norm_col = f"{col}_NORM"
    df_clean[norm_col] = df_clean[col].apply(normalize_text)
    df_clean[f"{col}_MISSING"] = (df_clean[norm_col].str.len() == 0).astype("int8")

    if col in COMPILED_MAP:
        cat_cols = []
        for cat, pats in COMPILED_MAP[col].items():
            outcol = f"{col}__{cat}"
            df_clean[outcol] = df_clean[norm_col].apply(lambda t: flag_any(t, pats)).astype("int8")
            cat_cols.append(outcol)

        df_clean[f"{col}__other"] = (
            (df_clean[f"{col}_MISSING"] == 0) & (df_clean[cat_cols].sum(axis=1) == 0)
        ).astype("int8")

# Save engineered comorbidity features ONLY (exclude manufacturer one-hot columns)
engineered_cols = [
    c for c in df_clean.columns
    if (("__" in c and not c.startswith("MANU__")) or c.endswith("_MISSING"))
]

comorb_df = df_clean[["VAERS_ID"] + engineered_cols].copy()
comorb_csv_path = os.path.join(OUTDIR, "comorbidity_indicators.csv")
comorb_df.to_csv(comorb_csv_path, index=False)
print("Saved:", comorb_csv_path, "shape:", comorb_df.shape)

In [ ]:
# ============================================================
# 16. COMORBIDITY DIAGNOSTICS
# ============================================================

indicator_cols = [c for c in df_clean.columns if "__" in c]

summary = (
    df_clean[indicator_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("count")
    .to_frame()
)
summary["percent"] = summary["count"] / len(df_clean) * 100
print(summary.head(40))

fields = ["HISTORY","CUR_ILL","ALLERGIES","PRIOR_VAX","LAB_DATA","OTHER_MEDS"]
coverage_results = []

for field in fields:
    if f"{field}_MISSING" not in df_clean.columns:
        continue
    field_rows = df_clean[df_clean[f"{field}_MISSING"] == 0]
    category_cols = [c for c in df_clean.columns if c.startswith(field + "__") and "other" not in c]
    if not category_cols:
        continue
    categorized = (field_rows[category_cols].sum(axis=1) > 0).mean()
    coverage_results.append({
        "field": field,
        "coverage_percent": categorized * 100,
        "sample_size": len(field_rows)
    })

coverage_df = pd.DataFrame(coverage_results)
print("\nCoverage by field:")
print(coverage_df)

## 7. Symptom text preprocessing

Three symptom text versions are created:
- `SYMPTOM_TEXT_CLEAN`: standard cleaned text  
- `SYMPTOM_TEXT_CLEAN_NOLEAK`: supervised-ML version with explicit outcome terms removed  
- `SYMPTOM_TEXT_CLEAN_CLUSTER`: clustering version with administrative/report-template terms removed

In [ ]:
# ============================================================
# 17. SYMPTOM TEXT PREPROCESSING
# ============================================================

_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# Outcome leakage terms for supervised prediction
LEAKAGE_PATTERNS = [re.compile(p) for p in [
    r"\bhospitali[sz]ed\b", r"\bicu\b", r"\ber\b", r"\bemergency\b",
    r"\bdeath\b", r"\bdied\b", r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b", r"\bventilat\w*\b"
]]

def scrub_leakage_terms(x):
    if not x:
        return ""
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    return _ws2.sub(" ", x).strip()

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = df_clean["SYMPTOM_TEXT_CLEAN"].apply(scrub_leakage_terms)

# Administrative/report-template terms for clustering cleanup
admin_patterns = [
    r"\blot number\b", r"\bbatch number\b", r"\broute administration\b",
    r"\bunspecified route\b", r"\bdosage form\b", r"\bform\b",
    r"\bintramuscular\b", r"\bpatient received\b", r"\breported\b",
    r"\bcase\b", r"\bunknown date\b", r"\bdate patient\b", r"\breporter\b",
    r"\bfollowing information\b", r"\badmin\b", r"\bprophylactic vaccination\b",
    r"\bvaccine lot\b", r"\bsingle dose\b", r"\bnumber unknown\b",
    r"\bcontactable\b", r"\bdescribed\b", r"\bsubject experienced\b", r"\bsubject\b",
    r"\breceived covid\b", r"\bcovid vaccine\b", r"\bvaccine\b",
    r"\bmoderna\b", r"\bpfizer\b", r"\bpfizer-biontech\b", r"\bbiontech\b",
    r"\bjanssen\b", r"\bnovavax\b", r"\bmrna\b", r"\bbnt\b"
]
admin_regexes = [re.compile(p) for p in admin_patterns]

def remove_admin_terms(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text).lower()
    for rgx in admin_regexes:
        text = rgx.sub(" ", text)
    text = _ws2.sub(" ", text).strip()
    return text

df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"] = df_clean["SYMPTOM_TEXT_CLEAN"].apply(remove_admin_terms)

# Save cleaned text variants
df_clean[["VAERS_ID", "SYMPTOM_TEXT_CLEAN", "SYMPTOM_TEXT_CLEAN_NOLEAK", "SYMPTOM_TEXT_CLEAN_CLUSTER"]].to_csv(
    f"{OUTDIR}/symptom_text_clean_variants.csv", index=False
)

print([c for c in df_clean.columns if "SYMPTOM" in c])

## 8. Unsupervised clustering: phenotype discovery

This section uses:
- `SYMPTOM_TEXT_CLEAN_CLUSTER`
- TF-IDF
- Truncated SVD / LSA
- MiniBatchKMeans
- sampled silhouette and ARI-based stability

In [ ]:
# ============================================================
# 18. CLUSTERING PREP: TF-IDF + SVD ON CLUSTER-CLEANED TEXT
# ============================================================

tfidf_cluster = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.9,
    stop_words="english",
    sublinear_tf=True,
    norm="l2",
    strip_accents="unicode",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
)

X_tfidf_cluster = tfidf_cluster.fit_transform(df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"])
print("TF-IDF shape:", X_tfidf_cluster.shape)

# Save TF-IDF + vocab
sp.save_npz(f"{OUTDIR}/symptom_text_tfidf_cluster_cleaned.npz", X_tfidf_cluster)
pd.DataFrame({"term": tfidf_cluster.get_feature_names_out()}).to_csv(
    f"{OUTDIR}/tfidf_vocabulary_cluster_cleaned.csv", index=False
)

# SVD / LSA
svd_cluster = TruncatedSVD(n_components=200, random_state=42)
X_lsa_cluster = svd_cluster.fit_transform(X_tfidf_cluster)
X_lsa_cluster = Normalizer(copy=False).fit_transform(X_lsa_cluster)

np.save(f"{OUTDIR}/symptom_text_lsa_200d_cluster_cleaned.npy", X_lsa_cluster)
np.save(f"{OUTDIR}/svd_components_200d_cluster_cleaned.npy", svd_cluster.components_)

print("X_lsa_cluster shape:", X_lsa_cluster.shape)

In [ ]:
# ============================================================
# 19. MODEL SELECTION: ELBOW + SAMPLED SILHOUETTE
# ============================================================

rng = np.random.default_rng(42)
silhouette_sample_size = 20000
silhouette_idx = rng.choice(X_lsa_cluster.shape[0], size=silhouette_sample_size, replace=False)
X_sil = X_lsa_cluster[silhouette_idx]

k_values = [5, 8, 10, 12, 15]
elbow_results = []

for k in k_values:
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        batch_size=10000,
        n_init=3
    )
    labels_full = km.fit_predict(X_lsa_cluster)
    labels_sil = labels_full[silhouette_idx]
    sil = silhouette_score(X_sil, labels_sil)

    elbow_results.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": sil
    })

    print(f"k={k:2d} | silhouette={sil:.4f} | inertia={km.inertia_:.2e}")

elbow_df = pd.DataFrame(elbow_results)
elbow_df.to_csv(f"{OUTDIR}/cluster_selection_metrics_cluster_cleaned.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(elbow_df["k"], elbow_df["inertia"], marker="o")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")
ax[1].plot(elbow_df["k"], elbow_df["silhouette"], marker="o")
ax[1].set_title("Sampled Silhouette")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 20. STABILITY ANALYSIS (ARI ACROSS RANDOM SEEDS)
# ============================================================

# Set chosen k here after reviewing elbow + silhouette
best_k = 8

stability_sample_size = 50000
stability_idx = rng.choice(X_lsa_cluster.shape[0], size=stability_sample_size, replace=False)
X_stability = X_lsa_cluster[stability_idx]

seeds = [1, 7, 21, 42, 88]
labels_by_seed = {}

for seed in seeds:
    km = MiniBatchKMeans(
        n_clusters=best_k,
        random_state=seed,
        batch_size=10000,
        n_init=3
    )
    labels_by_seed[seed] = km.fit_predict(X_stability)

stability_results = []
for i, seed_a in enumerate(seeds):
    for seed_b in seeds[i+1:]:
        ari = adjusted_rand_score(labels_by_seed[seed_a], labels_by_seed[seed_b])
        stability_results.append({"seed_a": seed_a, "seed_b": seed_b, "ARI": ari})

stability_df = pd.DataFrame(stability_results)
print(stability_df)
print("\nMean ARI:", stability_df["ARI"].mean())

stability_df.to_csv(f"{OUTDIR}/cluster_stability_ari_cluster_cleaned.csv", index=False)

In [ ]:
# ============================================================
# 21. FINAL CLUSTERING MODEL ON FULL DATASET
# ============================================================

final_km = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=42,
    batch_size=10000,
    n_init=5
)

df_clean["CLUSTER"] = final_km.fit_predict(X_lsa_cluster)

cluster_sizes = df_clean["CLUSTER"].value_counts().sort_index()
print(cluster_sizes)
(cluster_sizes / len(df_clean)).rename("proportion").to_csv(f"{OUTDIR}/cluster_sizes_cluster_cleaned.csv")

In [ ]:
# ============================================================
# 22. INTERPRET CLUSTERS USING RECONSTRUCTED TERM WEIGHTS
# ============================================================

feature_names = pd.read_csv(f"{OUTDIR}/tfidf_vocabulary_cluster_cleaned.csv")["term"].values
svd_components = np.load(f"{OUTDIR}/svd_components_200d_cluster_cleaned.npy")

reconstructed_centroids = final_km.cluster_centers_ @ svd_components

top_terms_rows = []
for cluster_id in range(best_k):
    centroid = reconstructed_centroids[cluster_id]
    top_idx = np.argsort(centroid)[::-1][:20]
    top_terms = feature_names[top_idx]

    print(f"\nCluster {cluster_id} top terms:")
    print(", ".join(top_terms))

    for rank, term in enumerate(top_terms, start=1):
        top_terms_rows.append({"cluster": cluster_id, "rank": rank, "term": term})

top_terms_df = pd.DataFrame(top_terms_rows)
top_terms_df.to_csv(f"{OUTDIR}/cluster_top_terms_cluster_cleaned.csv", index=False)

In [ ]:
# ============================================================
# 23. CLUSTER CHARACTERIZATION
# ============================================================

# Cluster vs serious outcome
cluster_serious = pd.crosstab(df_clean["CLUSTER"], df_clean["SERIOUS"], normalize="index")
print("\nCluster vs SERIOUS")
print(cluster_serious)
cluster_serious.to_csv(f"{OUTDIR}/cluster_vs_serious_cluster_cleaned.csv")

# Cluster vs sex
if "SEX" in df_clean.columns:
    cluster_sex = pd.crosstab(df_clean["CLUSTER"], df_clean["SEX"], normalize="index")
    print("\nCluster vs SEX")
    print(cluster_sex)
    cluster_sex.to_csv(f"{OUTDIR}/cluster_vs_sex_cluster_cleaned.csv")

# Cluster vs age
if "AGE_YRS" in df_clean.columns:
    cluster_age = df_clean.groupby("CLUSTER")["AGE_YRS"].describe()
    print("\nCluster age summary")
    print(cluster_age)
    cluster_age.to_csv(f"{OUTDIR}/cluster_age_summary_cluster_cleaned.csv")

# Cluster vs manufacturer
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]
if manufacturer_cols:
    cluster_manu = df_clean.groupby("CLUSTER")[manufacturer_cols].mean()
    print("\nCluster vs manufacturer")
    print(cluster_manu)
    cluster_manu.to_csv(f"{OUTDIR}/cluster_vs_manufacturer_cluster_cleaned.csv")

# Cluster vs comorbidities
comorb_cols = [c for c in df_clean.columns if "__" in c and not c.startswith("MANU__")]
if comorb_cols:
    cluster_comorb = df_clean.groupby("CLUSTER")[comorb_cols].mean()
    print("\nCluster vs comorbidities (first 10 cols)")
    print(cluster_comorb.iloc[:, :10])
    cluster_comorb.to_csv(f"{OUTDIR}/cluster_vs_comorbidities_cluster_cleaned.csv")

# Cluster summary
cluster_summary = df_clean.groupby("CLUSTER").agg({
    "SERIOUS": "mean",
    "AGE_YRS": "mean",
    "VAERS_ID": "count"
}).rename(columns={"SERIOUS": "SERIOUS_RATE", "AGE_YRS": "MEAN_AGE", "VAERS_ID": "N"})

print("\nCluster summary")
print(cluster_summary)
cluster_summary.to_csv(f"{OUTDIR}/cluster_summary_cluster_cleaned.csv")

## 9. Supervised severity prediction

This section evaluates four supervised models:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

All preprocessing (TF-IDF and SVD) is performed inside the cross-validation pipeline to avoid data leakage.

In [ ]:
# ============================================================
# 24. SUPERVISED MODEL SETUP
# ============================================================

# Choose features
TEXT_FEATURE = "SYMPTOM_TEXT_CLEAN_NOLEAK"
y = df_clean["SERIOUS"].astype(int)

numeric_cols = [c for c in ["AGE_YRS", "ONSET_INTERVAL", "NUMDAYS", "HOSPDAYS", "MAX_DOSE"] if c in df_clean.columns]
categorical_cols = [c for c in ["SEX", "STATE", "YEAR", "MONTH"] if c in df_clean.columns]
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]
dose_cols = [c for c in ["DOSE_COUNT", "MULTI_DOSE", "UNKNOWN_DOSE", "MULTI_MANUFACTURER"] if c in df_clean.columns]
comorb_cols = [c for c in df_clean.columns if "__" in c and not c.startswith("MANU__")]
missing_cols = [c for c in df_clean.columns if c.endswith("_MISSING") and c not in comorb_cols]

structured_cols = numeric_cols + categorical_cols + manufacturer_cols + dose_cols + comorb_cols + missing_cols

print("Numeric cols:", numeric_cols)
print("Categorical cols:", categorical_cols)
print("Structured feature count:", len(structured_cols))

In [ ]:
# ============================================================
# 25. PREPROCESSORS + MODELS
# ============================================================

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

structured_preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
        ("passthrough_binary", "passthrough", [c for c in structured_cols if c not in numeric_cols + categorical_cols]),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

text_preprocess = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        min_df=10,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True,
        norm="l2",
        strip_accents="unicode",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
    )),
    ("svd", TruncatedSVD(n_components=200, random_state=42))
])

full_preprocess = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess, structured_cols),
        ("text", text_preprocess, TEXT_FEATURE),
    ],
    remainder="drop"
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

param_grids = {
    "Logistic Regression": {"clf__C": [0.1, 1.0, 3.0]},
    "Decision Tree": {"clf__max_depth": [5, 10, 20], "clf__min_samples_split": [2, 5, 10]},
    "Random Forest": {"clf__n_estimators": [100, 200], "clf__max_depth": [10, 20], "clf__min_samples_split": [2, 5]},
    "Gradient Boosting": {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1], "clf__max_depth": [3, 5]}
}

In [ ]:
# ============================================================
# 26. STRATIFIED K-FOLD CV + GRID SEARCH
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, clf in models.items():
    print("\n" + "="*80)
    print("MODEL:", name)
    print("="*80)

    pipe = Pipeline([
        ("preprocess", full_preprocess),
        ("clf", clf)
    ])

    grid = GridSearchCV(
        pipe,
        param_grids[name],
        scoring="average_precision",  # PR-AUC
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(df_clean, y)
    best = grid.best_estimator_
    print("Best params:", grid.best_params_)

    y_pred = cross_val_predict(best, df_clean, y, cv=cv, method="predict", n_jobs=-1)
    y_prob = cross_val_predict(best, df_clean, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

    pr_auc = average_precision_score(y, y_prob)
    f1 = f1_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    results.append({
        "Model": name,
        "PR-AUC": pr_auc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "ConfusionMatrix": cm
    })

    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print("Confusion Matrix:")
    print(cm)

summary_df = pd.DataFrame([{k:v for k,v in r.items() if k != "ConfusionMatrix"} for r in results])
print("\nMODEL COMPARISON")
print(summary_df.sort_values("PR-AUC", ascending=False))

summary_df.to_csv(f"{OUTDIR}/supervised_model_comparison.csv", index=False)

## 10. Save the final enriched dataset

In [ ]:
# ============================================================
# 27. SAVE FINAL DATASET
# ============================================================

final_csv_path = f"{OUTDIR}/vaers_dataset_with_features.csv"
df_clean.to_csv(final_csv_path, index=False)
print("Saved final dataset:", final_csv_path, "shape:", df_clean.shape)

# Optional parquet save if engine is available
try:
    final_parquet_path = f"{OUTDIR}/vaers_dataset_with_features.parquet"
    df_clean.to_parquet(final_parquet_path, index=False)
    print("Saved parquet:", final_parquet_path)
except Exception as e:
    print("Parquet not saved (likely missing pyarrow/fastparquet). CSV is saved.")
    print("Parquet error:", repr(e))

# End of notebook

This notebook is now organized so that:
- comorbidity indicators do **not** duplicate manufacturer columns
- clustering uses `SYMPTOM_TEXT_CLEAN_CLUSTER`
- supervised models use `SYMPTOM_TEXT_CLEAN_NOLEAK`
- all major artifacts are saved to disk